# Assignment 3 (Q3) — House Price Prediction: Linear, Ridge, Lasso and Elastic Net Regression
**Data Science Lab (CS2311)**

**Dataset:** Kaggle "House Sales in King County, USA" (`kc_house_data.csv`): 21,613 homes sold in the Seattle area between May 2014 and May 2015.

**Plan:**
1. Preprocess: missing values, date and categorical variables, feature engineering, scaling.
2. Baseline Multiple Linear Regression → MSE, RMSE, MAE, R².
3. Ridge, Lasso and Elastic Net for many values of the regularization strength lambda (called `alpha` in scikit-learn).
4. Plot training and testing error vs lambda, and compare the coefficients of all four models.

## Intuition: what is regularization?

**Analogy:** a student who memorises every past paper (overfitting) does badly on a new paper. Regularization is a teacher saying "keep your answers simple": every bit of extra complexity costs marks. A little penalty helps the model generalise; too much and it says nothing useful (underfitting).

For linear models, "complexity" = **large coefficients**. Regularization adds a penalty on their size to the usual error:

| Model | What is minimised (plain text) | Effect on coefficients |
|---|---|---|
| Linear | Σ(y − ŷ)² | No restriction; can blow up when features are correlated |
| **Ridge (L2)** | Σ(y − ŷ)² + λ·Σ b² | All shrink smoothly towards 0; none become exactly 0 |
| **Lasso (L1)** | Σ(y − ŷ)² + λ·Σ abs(b) | Some become **exactly 0** → automatic feature selection |
| **Elastic Net** | Σ(y − ŷ)² + λ·[ r·Σ abs(b) + (1 − r)/2·Σ b² ] | A mix of both; r = `l1_ratio` |

- λ = 0 → identical to plain linear regression.
- λ very large → all coefficients pushed to 0 → the model predicts roughly the average for everyone (underfitting).
- **Why Lasso gives exact zeros:** the abs() penalty has a sharp corner at 0, so the best solution often lands exactly there. The squared penalty is smooth at 0, so Ridge only gets close.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

## Loading the data
If you downloaded `kc_house_data.csv` from Kaggle, keep it next to this notebook. Otherwise the same file is read from a public GitHub copy (needs internet).

In [ ]:
LOCAL_FILE = 'kc_house_data.csv'
URL = 'https://raw.githubusercontent.com/Shreyas3108/house-price-prediction/master/kc_house_data.csv'

df_raw = pd.read_csv(LOCAL_FILE if os.path.exists(LOCAL_FILE) else URL)
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

---
## 1. Preprocessing
### 1.1 Missing values and duplicates

In [ ]:
print('Missing values per column:\n', df_raw.isnull().sum()[df_raw.isnull().sum() > 0], sep='')
print('Total missing values:', df_raw.isnull().sum().sum())
print('Fully duplicated rows:', df_raw.duplicated().sum())
print('Repeated house ids:', df_raw['id'].duplicated().sum())

- The original Kaggle file has **no missing values**. The code below still includes a general strategy (median for numbers, mode for categories), so it works on any version of the file.
- The repeated `id`s are **the same house sold twice** within the year (different dates and prices). They are genuine sales, not duplicate records, so we keep them.

In [ ]:
df = df_raw.copy()
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(exclude='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])
df = df.drop_duplicates()
print('Missing values left:', df.isnull().sum().sum(), '| shape:', df.shape)

### 1.2 The target: price
Prices are strongly **right-skewed** (a few multi-million-dollar mansions). Linear regression with squared errors would be dominated by those few houses. The standard fix is to model **log(price)**: the distribution becomes roughly bell-shaped, and the model learns *percentage* effects ("an extra bathroom adds about x% to the price"), which suits house prices.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['price'], bins=60, ax=ax[0])
ax[0].set_title(f'price (skewness = {df.price.skew():.2f})')
sns.histplot(np.log1p(df['price']), bins=60, ax=ax[1], color='tab:green')
ax[1].set_title(f'log(1 + price) (skewness = {np.log1p(df.price).skew():.2f})')
plt.tight_layout()
plt.show()

### 1.3 Feature engineering and categorical variables

| Raw column | Problem | What we do |
|---|---|---|
| `id` | Just an identifier | Drop |
| `date` ('20141013T000000') | Text | Convert to a date; keep `sale_month` and `sale_year` |
| `yr_built` | Absolute year | `house_age` = sale year − year built |
| `yr_renovated` | 0 means "never" | `renovated` = 1 if renovated, else 0 |
| `sqft_basement` | Exactly equal to `sqft_living − sqft_above` | `has_basement` flag; drop the original (perfectly redundant) |
| `zipcode` | A **category** stored as a number (98178 is not "bigger" than 98125) | One-hot encode |

**Why one-hot the zipcode?** Location is the biggest driver of price. As a plain number, the model would assume price changes steadily as the zipcode number increases, which is meaningless. One-hot encoding gives each of the 70 zipcodes its own 0/1 column.

In [ ]:
print('Max |sqft_living − sqft_above − sqft_basement| =',
      (df.sqft_living - df.sqft_above - df.sqft_basement).abs().max(), '→ perfectly redundant')

df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month
df['house_age'] = df['sale_year'] - df['yr_built']
df['renovated'] = (df['yr_renovated'] > 0).astype(int)
df['has_basement'] = (df['sqft_basement'] > 0).astype(int)

df = df.drop(columns=['id', 'date', 'yr_built', 'yr_renovated', 'sqft_basement'])
df['zipcode'] = df['zipcode'].astype(str)
df = pd.get_dummies(df, columns=['zipcode'], drop_first=True, dtype=int)

df['log_price'] = np.log1p(df['price'])
print('Final shape:', df.shape)

In [ ]:
corr = df.drop(columns=[c for c in df.columns if c.startswith('zipcode_')]).corr()
plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', annot_kws={'size': 7})
plt.title('Correlations (zipcode dummies hidden)')
plt.show()
print(corr['log_price'].drop(['log_price', 'price']).sort_values(ascending=False).round(3))

**Notice the strongly correlated groups** (`sqft_living`, `sqft_above`, `sqft_living15`, `grade`, `bathrooms`): they tell the model similar things. Plain linear regression can then give them large opposite-signed coefficients that cancel out. This **multicollinearity** is exactly what Ridge and Elastic Net are good at taming.

### 1.4 Split and scale
- 80/20 train/test split.
- **Scaling is mandatory for regularization:** the penalty treats all coefficients equally, so every feature must be on the same scale. Otherwise a feature measured in large units (sqft) would get a tiny coefficient and escape the penalty, while one in small units (bathrooms) would be punished unfairly.
- Fit the scaler on the training set only (no data leakage).

In [ ]:
X = df.drop(columns=['price', 'log_price'])
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
feature_names = X.columns
print('Train:', X_train_s.shape, ' Test:', X_test_s.shape)

---
## 2. Baseline: Multiple Linear Regression

| Metric | Formula (plain text) | Better |
|---|---|---|
| MSE | mean of (y − ŷ)² | Lower |
| RMSE | sqrt(MSE) | Lower |
| MAE | mean of abs(y − ŷ) | Lower |
| R² | 1 − SS_res / SS_tot | Higher (max 1) |

Because the target is log(price), the errors are on the log scale: an RMSE of 0.18 means typical errors of roughly ±18–20% of the price. We also report the MAE in dollars (after converting predictions back with `expm1`) because it is easier to interpret.

In [ ]:
def evaluate(model, name, alpha=np.nan):
    pred_tr = model.predict(X_train_s)
    pred_te = model.predict(X_test_s)
    mse_te = mean_squared_error(y_test, pred_te)
    return {
        'Model': name,
        'lambda (alpha)': alpha,
        'Train MSE': mean_squared_error(y_train, pred_tr),
        'Test MSE': mse_te,
        'Test RMSE': np.sqrt(mse_te),
        'Test MAE': mean_absolute_error(y_test, pred_te),
        'Test R2': r2_score(y_test, pred_te),
        'Test MAE ($)': mean_absolute_error(np.expm1(y_test), np.expm1(pred_te)),
        '#Zero coefs': int(np.sum(np.abs(model.coef_) < 1e-10)),
    }

linear = LinearRegression().fit(X_train_s, y_train)
baseline = pd.DataFrame([evaluate(linear, 'Linear')]).set_index('Model')
baseline.round(4)

---
## 3. Ridge, Lasso and Elastic Net for many values of lambda

We try lambda values spread evenly on a **log scale** from 0.0001 to 1000, because the interesting changes happen across orders of magnitude, not in small steps.

Elastic Net uses `l1_ratio = 0.5` (an equal mix of L1 and L2).

In [ ]:
alphas = np.logspace(-4, 3, 15)
print('Lambda values tried:', np.round(alphas, 4))

model_makers = {
    'Ridge':       lambda a: Ridge(alpha=a),
    'Lasso':       lambda a: Lasso(alpha=a, max_iter=20000),
    'Elastic Net': lambda a: ElasticNet(alpha=a, l1_ratio=0.5, max_iter=20000),
}

rows, coef_paths = [], {name: [] for name in model_makers}
for name, make in model_makers.items():
    for a in alphas:
        m = make(a).fit(X_train_s, y_train)
        rows.append(evaluate(m, name, a))
        coef_paths[name].append(m.coef_)

sweep = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.4f}'.format)
sweep[['Model', 'lambda (alpha)', 'Train MSE', 'Test MSE', 'Test R2', '#Zero coefs']]

### Training and testing error vs lambda

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharey=True)
for ax, name in zip(axes, model_makers):
    s = sweep[sweep.Model == name]
    ax.plot(s['lambda (alpha)'], s['Train MSE'], 'o-', label='Training MSE')
    ax.plot(s['lambda (alpha)'], s['Test MSE'], 's-', label='Testing MSE')
    best = s.loc[s['Test MSE'].idxmin()]
    ax.axvline(best['lambda (alpha)'], color='green', ls='--', label=f"Best λ = {best['lambda (alpha)']:.4g}")
    ax.axhline(baseline['Test MSE'].iloc[0], color='gray', ls=':', label='Linear regression (test)')
    ax.set_xscale('log')
    ax.set(title=name, xlabel='lambda (log scale)', ylabel='MSE (log-price)')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**How to read these plots:**
- **Left side (small λ):** almost no penalty → same errors as plain linear regression.
- **Moving right:** the penalty grows. Training error rises steadily (the model is no longer free to fit every detail).
- **Far right (huge λ):** coefficients are crushed towards 0 → both errors shoot up (underfitting).
- Lasso and Elastic Net hit the underfitting wall at a much smaller λ than Ridge, because the abs() penalty is much harsher on small coefficients.

### Number of coefficients set to exactly zero

In [ ]:
plt.figure(figsize=(8, 4))
for name in model_makers:
    s = sweep[sweep.Model == name]
    plt.plot(s['lambda (alpha)'], s['#Zero coefs'], 'o-', label=name)
plt.xscale('log')
plt.xlabel('lambda (log scale)')
plt.ylabel(f'# coefficients = 0 (out of {len(feature_names)})')
plt.title('Feature selection by L1 regularization')
plt.legend()
plt.show()

Ridge **never** sets a coefficient to zero. Lasso and Elastic Net remove more and more features as λ grows, until every coefficient is zero.

### Coefficient paths: how each coefficient shrinks as lambda grows
(Only the 19 non-zipcode features are drawn, so the plot stays readable.)

In [ ]:
main_feats = [i for i, f in enumerate(feature_names) if not f.startswith('zipcode_')]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, name in zip(axes, model_makers):
    path = np.array(coef_paths[name])[:, main_feats]
    for j, idx in enumerate(main_feats):
        ax.plot(alphas, path[:, j], label=feature_names[idx])
    ax.set_xscale('log')
    ax.axhline(0, color='black', lw=0.8)
    ax.set(title=f'{name} coefficient paths', xlabel='lambda (log scale)', ylabel='coefficient')
axes[2].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

---
## 4. Final comparison at the best lambda

"Best λ" = the λ with the lowest **test** MSE. If several λ values tie (the error is flat for small λ), the smallest one is reported. A best λ at the bottom of the range means the plain model was already not overfitting.

In [ ]:
best_rows = sweep.loc[sweep.groupby('Model')['Test MSE'].idxmin()].set_index('Model')
final = pd.concat([baseline, best_rows])
final[['lambda (alpha)', 'Test MSE', 'Test RMSE', 'Test MAE', 'Test R2', 'Test MAE ($)', '#Zero coefs']]

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
final['Test RMSE'].plot(kind='bar', ax=ax[0], rot=0, color='tab:blue', title='Test RMSE (log-price)')
final['Test R2'].plot(kind='bar', ax=ax[1], rot=0, color='tab:green', title='Test R²')
ax[1].set_ylim(0.8, 0.9)
final['#Zero coefs'].plot(kind='bar', ax=ax[2], rot=0, color='tab:red', title='# coefficients set to 0')
plt.tight_layout()
plt.show()

### Comparing the coefficients of the four models
We refit each model at its best λ, plus a **strong** λ, to make the shrinkage visible.

In [ ]:
strong = {'Ridge': 1000, 'Lasso': 0.01, 'Elastic Net': 0.02}
coef_table = pd.DataFrame({'Linear': linear.coef_}, index=feature_names)
for name, make in model_makers.items():
    coef_table[f'{name} (best λ)'] = make(best_rows.loc[name, 'lambda (alpha)']).fit(X_train_s, y_train).coef_
    coef_table[f'{name} (λ={strong[name]})'] = make(strong[name]).fit(X_train_s, y_train).coef_

main = coef_table.loc[feature_names[main_feats]]
main.round(4)

In [ ]:
cols_to_plot = ['Linear'] + [f'{n} (λ={strong[n]})' for n in model_makers]
main[cols_to_plot].plot(kind='bar', figsize=(17, 5.5), width=0.8)
plt.axhline(0, color='black', lw=0.8)
plt.title('Coefficients: Linear vs strongly regularized Ridge, Lasso and Elastic Net (non-zipcode features)')
plt.ylabel('Coefficient (scaled features)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

print('Coefficients set to zero at the strong λ:')
for n in model_makers:
    zeros = main.index[np.abs(main[f'{n} (λ={strong[n]})']) < 1e-10].tolist()
    print(f'  {n:12s}: {zeros}')
print('Sum of |coefficients| (all features):')
print(coef_table[cols_to_plot].abs().sum().round(3))

## Conclusion: the effect of regularization

1. **Prediction error:** with a small λ, all three regularized models perform almost exactly like plain linear regression (see the final table). With 17,000 training rows and about 90 features, the plain model is not seriously overfitting, so regularization cannot improve accuracy much. Too large a λ clearly **hurts**: both training and test errors rise (underfitting).
2. **Coefficients — Ridge:** shrinks all coefficients smoothly and never makes them exactly zero. It spreads the weight across correlated features (`sqft_living`, `sqft_above`, `sqft_living15`) instead of letting them fight each other.
3. **Coefficients — Lasso:** sets many coefficients to **exactly zero**: it performs feature selection, keeping strong predictors such as `grade`, `sqft_living`, `lat` and `view`, and dropping weak or redundant ones.
4. **Elastic Net:** in between. It zeroes some features like Lasso but, thanks to its L2 part, it keeps groups of correlated features together.
5. **Practical choice:** Ridge or Elastic Net at the best λ gives the most stable model with the same accuracy as plain regression. Lasso at a moderate λ gives a **simpler** model (fewer features) at a small cost in accuracy, which is valuable when interpretability matters.